# Lab 1 · O Decisor
**Oficina do Builder · Winovoito** — regra, ML, LLM, cadeia ou agente?

Neste lab você:
1. transforma a Ficha de decisão numa regra em Python;
2. testa a regra contra os 6 casos da aula 1.3;
3. usa um modelo de linguagem para preencher a ficha a partir de texto livre;
4. compara o que o modelo preencheu com o gabarito e anota onde ele erra.

> Antes de começar, guarde `OFICINA_API_KEY`, `OFICINA_BASE_URL` e `OFICINA_MODELO` no cofre de segredos do Colab (ícone de chave na barra lateral). **Nunca escreva a chave numa célula.** As partes 1 e 2 funcionam sem chave nenhuma.

In [ ]:
# Setup: baixa o material aberto da Oficina (rode uma vez)
REPO_URL = "https://github.com/winovoito/oficina-do-builder-aberto"  # endereço definitivo a confirmar
import os, sys
if not os.path.exists("oficina-do-builder-aberto"):
    !git clone -q {REPO_URL}
sys.path.insert(0, "oficina-do-builder-aberto")
import json
from pathlib import Path
from oficina.decisor import Criterios, Decisao
CASOS = json.loads(Path("oficina-do-builder-aberto/dados/casos.json").read_text("utf-8"))
print(f"{len(CASOS)} casos carregados")

## Parte 1 · A ficha vira regra

Complete a função `decidir`. Ela recebe os critérios e devolve uma `Decisao(abordagem, justificativa, salvaguardas)`.

Abordagens possíveis: `"regra"`, `"ml"`, `"llm"`, `"cadeia"`, `"agente"`.

Dicas da aula 1.1: se o caminho de várias etapas varia, agente; se é fixo, cadeia; se a lógica cabe numa frase, regra; tabela com histórico rotulado suficiente, ML; texto livre, LLM. E não esqueça as salvaguardas quando o custo do erro é alto.

In [ ]:
def decidir(c: Criterios) -> Decisao:
    # TODO: escreva a sua regra aqui
    salvaguardas = []
    abordagem = "llm"
    justificativa = "TODO"
    return Decisao(abordagem, justificativa, salvaguardas)

## Parte 2 · Testes antes de confiar

In [ ]:
def testar(funcao_decidir):
    acertos = 0
    for caso in CASOS:
        d = funcao_decidir(Criterios(**caso["criterios"]))
        ok = d.abordagem == caso["esperado"]["abordagem"]
        acertos += ok
        print(f"{'✅' if ok else '❌'} {caso['nome']}: {d.abordagem} (esperado: {caso['esperado']['abordagem']})")
    print(f"\n{acertos}/{len(CASOS)} casos")

testar(decidir)

### Gabarito
Só abra depois de tentar. A solução de referência está em `oficina/decisor.py`.

In [ ]:
from oficina.decisor import decidir as decidir_referencia
testar(decidir_referencia)

## Parte 3 · O LLM preenche a ficha

O prompt está num arquivo próprio, `prompts/extrair_criterios_v1.md`. Leia antes de rodar. Repare em duas coisas: a descrição vai delimitada, e o prompt diz que ela é **dado, não instrução**.

A divisão de trabalho: o LLM faz o que faz bem (entender texto) e a regra faz o que faz bem (decidir de forma previsível).

In [ ]:
print(Path("oficina-do-builder-aberto/prompts/extrair_criterios_v1.md").read_text("utf-8"))

In [ ]:
from oficina.extracao import extrair_criterios
extraidos = {}
for caso in CASOS:
    c = extrair_criterios(caso["descricao"])
    extraidos[caso["id"]] = c
    print(f"{caso['nome']} → {decidir_referencia(c).abordagem}")

## Parte 4 · Onde o LLM erra

In [ ]:
from dataclasses import asdict
total = iguais = 0
for caso in CASOS:
    gabarito, obtido = caso["criterios"], asdict(extraidos[caso["id"]])
    for campo, valor in gabarito.items():
        total += 1
        if campo in ("volume_exemplos", "decisoes_mes"):
            # números são estimativas: conta como acerto se estiver na mesma ordem de grandeza
            igual = (valor == 0) == (obtido[campo] == 0) and (valor == 0 or 0.1 <= obtido[campo] / valor <= 10)
        else:
            igual = obtido[campo] == valor
        iguais += igual
        if not igual:
            print(f"{caso['nome']} · {campo}: modelo={obtido[campo]!r} gabarito={valor!r}")
print(f"\nConcordância: {iguais}/{total} ({iguais/total:.0%})")

**Reflexão** (escreva aqui):
- Em que campo o modelo mais errou? Por quê?
- Algum erro mudaria a abordagem escolhida? Qual seria o custo disso?

## Parte 5 · Um caso seu

In [ ]:
meu_caso = """Descreva aqui um problema real (sem dados confidenciais)."""
c = extrair_criterios(meu_caso)
d = decidir_referencia(c)
print(c, "\n\n→", d.abordagem, "—", d.justificativa)
for s in d.salvaguardas:
    print("  •", s)

## Ao terminar
Rode o **Revisor dos 7 erros** (`revisor/revisor-7-erros.md`) numa conversa nova, com o seu notebook. Confira principalmente se nenhuma chave ficou escrita numa célula.

---
Winovoito · código Apache-2.0 · texto CC BY 4.0 · *Feito e revisado por IAs. Orquestrado por humanos.*